In [5]:
# !pip install torch torchvision pillow matplotlib huggingface_hub

import os
from PIL import Image

import torch
import torch.nn as nn
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

from huggingface_hub import hf_hub_download

In [17]:
MODEL_REPO = "GeraldNdawula/Watermark_Removal_UNet"
MODEL_FILE = "unet_final.pth"

INPUT_DIR = "/content/drive/MyDrive/Stat Assignments/Project/Des Evans "
OUTPUT_DIR = "des_evals_outputs"

IMAGE_SIZE = 128
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Using device:", DEVICE)

Using device: cuda


In [9]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, features=[64, 128, 256, 512]):
        super().__init__()

        self.encoders = nn.ModuleList()
        self.pools = nn.ModuleList()
        self.upconvs = nn.ModuleList()
        self.decoders = nn.ModuleList()

        current_channels = in_channels

        for feature in features:
            self.encoders.append(DoubleConv(current_channels, feature))
            self.pools.append(nn.MaxPool2d(kernel_size=2, stride=2))
            current_channels = feature

        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        for feature in reversed(features):
            self.upconvs.append(
                nn.ConvTranspose2d(feature * 2, feature, kernel_size=2, stride=2)
            )
            self.decoders.append(
                DoubleConv(feature * 2, feature)
            )

        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []

        for encoder, pool in zip(self.encoders, self.pools):
            x = encoder(x)
            skip_connections.append(x)
            x = pool(x)

        x = self.bottleneck(x)

        skip_connections = skip_connections[::-1]

        for idx in range(len(self.upconvs)):
            x = self.upconvs[idx](x)
            skip_connection = skip_connections[idx]

            if x.shape != skip_connection.shape:
                x = torch.nn.functional.interpolate(
                    x,
                    size=skip_connection.shape[2:],
                    mode="bilinear",
                    align_corners=False
                )

            x = torch.cat((skip_connection, x), dim=1)
            x = self.decoders[idx](x)

        return torch.tanh(self.final_conv(x))

In [10]:
model_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename="unet_final.pth"
)

ckpt = torch.load(model_path, map_location=DEVICE)

model = UNet().to(DEVICE)
model.load_state_dict(ckpt["model"])
model.eval()

print("Model loaded successfully.")

Model loaded successfully.


In [11]:
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.5] * 3)
])

def tensor_to_pil(tensor):
    tensor = tensor.squeeze(0).cpu()
    tensor = (tensor * 0.5) + 0.5
    tensor = tensor.clamp(0, 1)
    return transforms.ToPILImage()(tensor)

In [18]:
image_files = [
    f for f in os.listdir(INPUT_DIR)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
]

print("Images found:", len(image_files))
print(image_files)

Images found: 8
['IMG_7651.jpeg', 'IMG_7669.jpeg', 'IMG_7667.jpeg', 'IMG_7663.jpeg', 'IMG_7664.jpeg', 'IMG_7652.jpeg', 'IMG_7662.jpeg', 'IMG_7654.jpeg']


In [19]:
for file_name in image_files:
    img_path = os.path.join(INPUT_DIR, file_name)

    original_img = Image.open(img_path).convert("RGB")
    input_tensor = transform(original_img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        clean_tensor = model(input_tensor)

    output_img = tensor_to_pil(clean_tensor)

    save_path = os.path.join(
        OUTPUT_DIR,
        f"edited_{os.path.splitext(file_name)[0]}.png"
    )

    output_img.save(save_path)

    plt.figure(figsize=(8, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(original_img.resize((IMAGE_SIZE, IMAGE_SIZE)))
    plt.title("Watermarked Input")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(output_img)
    plt.title("Edited Output")
    plt.axis("off")

    plt.suptitle(file_name)
    plt.tight_layout()
    plt.show()

print("Finished running model.")
print("Edited images saved in:", OUTPUT_DIR)

Output hidden; open in https://colab.research.google.com to view.